# Fill Excel file information
Automatically parse folders added to the excel files with the experiments and add the relevant information. 

In [ ]:
#Analyse the Ryr1 myo15 excel and fill missing information automatically
import pandas as pd 
import os

# Change to the appropriate file
df = pd.read_excel("Ryr1-Myo15GCamp.xlsx")

df.dropna(subset=['Folder'], inplace=True)

In [ ]:
#Replace the media drive
drive = '/media/marcotti-lab'# C:, D:, Z:



df['Folder'] = drive + df['Folder'].str[2:]


In [3]:
#Load the xmfile and read the data

import xml.etree.ElementTree as ET
def parse_experiment_xml(xml_path):
    """
    Parse ThorImage Experiment.xml to extract imaging parameters.
    
    Returns dict with: pixelX, pixelY, pixelSizeUM, widthUM, heightUM, frameRate
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    lsm = root.find('LSM')
    
    return {
        'pixelX': int(lsm.get('pixelX')),          # 1024
        'pixelY': int(lsm.get('pixelY')),          # 288
        'pixelSizeUM': float(lsm.get('pixelSizeUM')),  # 0.72
        'widthUM': float(lsm.get('widthUM')),      # 245
        'heightUM': float(lsm.get('heightUM')),    # 68.91
        'frameRate': float(lsm.get('frameRate'))   # 27.163
    }
# Usage:
# params = parse_experiment_xml('Experiment.xml')
# print(params)



In [4]:
import sys
sys.path.append('src')
import movieTools as mt

In [ ]:
for j,el in df.iterrows():
    folder = el['Folder']
    xmlFile = os.path.join(folder,'Experiment.xml')
    imginfo = parse_experiment_xml(xmlFile)
    nFrames = mt.getImgInfo(folder)[2]

    if pd.isna(df.loc[j,'Pixel height']):
        df.loc[j,'Pixel height'] = imginfo['pixelY']
    if pd.isna(df.loc[j,'Pixel width']):
        df.loc[j,'Pixel width'] = imginfo['pixelX']
    if pd.isna(df.loc[j,'um width']):
        df.loc[j,'um width'] = imginfo['widthUM']
    if pd.isna(df.loc[j,'um/pixel']):
        df.loc[j,'um/pixel'] = imginfo['widthUM']/imginfo['pixelX']
    if pd.isna(df.loc[j,'fps']):
        df.loc[j,'fps'] = imginfo['frameRate']
    if  pd.isna(df.loc[j,'frames']):
        df.loc[j,'frames'] = nFrames
    if pd.isna(df.loc[j,'seconds']):
        df.loc[j,'seconds'] = nFrames/imginfo['frameRate']
    

In [ ]:
#Save to excel

#Change the drive back to the original
df['Folder'] = 'Z:' + df['Folder'].str[len(drive):]


#Note that the excel file is saved to a new file for safety. 
df.to_excel("Ryr1-Myo15GCamp_2.xlsx", index=False)
